# Template Praktikum — Klasifikasi Teks (PyTorch)

**Cara pakai: ubah `CFG` di §1, lalu Run All. Selesai.**

Kembaran neural dari `latihan_sklearn.ipynb` — bentuk config-nya sengaja dibuat mirip, hanya
bagian model yang berbeda. Penjelasan teorinya ada di `cheatsheet_pytorch.ipynb`.

| Bagian | Isi |
|---|---|
| §1 | **SEL CONFIG** |
| §2 | Menu: semua nilai yang sah |
| §3 | Mesin: validasi + load data (bentuk input sama seperti versi sklearn) |
| §4 | Mesin: preprocessing + vocab + tensor |
| §5 | Mesin: arsitektur model |
| §6 | Mesin: training loop + evaluasi |
| §7 | **JALANKAN** |
| §8 | Bandingkan beberapa config sekaligus |
| §9 | Pembanding TF-IDF (wajib ada di laporan) |
| §10 | Simpan model & prediksi teks baru |

> Ingat: pada data kecil, TF-IDF + LogReg sering menang. §9 menghitungnya otomatis supaya kamu
> punya angka pembanding yang jujur.

---
## §1 · SEL CONFIG  ·  ubah di sini saja

In [1]:
CFG = {
    # ---------------------------------------------------------------- 1. INPUT (sama seperti versi sklearn)
    "sumber":      "train_test",             # satu_file | train_test | folder_txt | dataframe
    "path":        "data/sentiment140_sample.csv",
    "path_train":  "data/split/train.csv",
    "path_test":   "data/split/test.csv",
    "text_col":    None,                     # None = deteksi otomatis
    "label_col":   None,
    "sep":         None,
    "header":      "infer",                  # "infer" | None
    "encoding":    None,
    "label_map":   {0: "negatif", 4: "positif"},
    "sample":      None,

    # ---------------------------------------------------------------- 2. CASE
    "case":        "biner",                  # biner | multiclass | multilabel | imbalanced
    "pemisah_label": "|",
    "bahasa":      "en",                     # en | id

    # ---------------------------------------------------------------- 3. PREPROCESSING
    "lowercase":         True,
    "mask_entity":       True,
    "buang_stopword":    False,              # untuk neural, stopword sering justru berguna
    "jaga_negasi":       True,
    "normalisasi_slang": False,
    "stemming":          False,

    # ---------------------------------------------------------------- 4. TEKS -> TENSOR
    "min_freq":    2,                        # kata dengan frekuensi < ini jadi <unk>
    "max_vocab":   20000,
    "max_len":     48,                       # token per dokumen, sisanya dipotong

    # ---------------------------------------------------------------- 5. MODEL
    "arsitektur":  "meanpool",               # meanpool | cnn | lstm | gru
    "embedding":   "acak",                   # acak | word2vec
    "freeze_emb":  False,                    # bekukan embedding pretrained
    "dim":         100,                      # dimensi embedding
    "hidden":      128,                      # unit LSTM/GRU
    "n_filter":    100,                      # filter per ukuran kernel (CNN)
    "kernel":      (3, 4, 5),                # ukuran kernel CNN
    "bidirectional": True,                   # LSTM/GRU dua arah
    "dropout":     0.3,

    # ---------------------------------------------------------------- 6. TRAINING
    "epochs":      6,
    "batch":       64,
    "lr":          1e-3,
    "weight_decay": 0.0,
    "seimbangkan": False,                    # bobot kelas pada loss (untuk case imbalanced)
    "val_size":    0.15,                     # validation dipotong dari train
    "test_size":   0.2,                      # dipakai kalau sumber != train_test
    "random_state": 42,
}

---
## §2 · Menu

### `arsitektur`
| Nilai | Model | Referensi slide | Catatan |
|---|---|---|---|
| `meanpool` | Embedding → rata-rata → Linear | 02b hal. 4 | paling cepat, baseline neural |
| `cnn` | Conv1d k=3,4,5 → max-pool | 02b hal. 10 (Kim 2014) | menangkap n-gram, cepat |
| `lstm` | LSTM (opsional dua arah) | 02b hal. 11 | memakai urutan kata, paling lambat |
| `gru` | GRU, versi ringkas LSTM | 02b hal. 11 | mirip LSTM, lebih cepat |

### `embedding`
| Nilai | Artinya |
|---|---|
| `acak` | vektor kata diinisialisasi acak dan dilatih dari nol |
| `word2vec` | dilatih dulu dengan gensim pada data latih (skip-gram), lalu dipakai sebagai bobot awal |

`freeze_emb=True` membekukan vektor (aman untuk data sangat kecil); `False` membiarkannya ikut
disetel — pada praktik biasanya `False` yang menang.

### Parameter yang berpengaruh ke tiap arsitektur
| Parameter | meanpool | cnn | lstm/gru |
|---|---|---|---|
| `dim` | ✔ | ✔ | ✔ |
| `hidden` | – | – | ✔ |
| `n_filter`, `kernel` | – | ✔ | – |
| `bidirectional` | – | – | ✔ |
| `dropout` | ✔ | ✔ | ✔ |

### Yang otomatis diurus notebook
- `case="multilabel"` → loss ganti ke `BCEWithLogitsLoss`, prediksi pakai ambang 0,5.
- `case="imbalanced"` + `seimbangkan=True` → bobot kelas dihitung dari frekuensi latih.
- Panjang kalimat < kernel terbesar → padding otomatis (kalau tidak, CNN error).
- GPU dipakai kalau ada; kalau tidak, jalan di CPU.

---
## §3 · Mesin: validasi + load data

In [2]:
import re, time, random, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

warnings.filterwarnings("ignore", category=UserWarning)
DF_MANUAL = None

PILIHAN = {
    "sumber":     {"satu_file", "train_test", "folder_txt", "dataframe"},
    "case":       {"biner", "multiclass", "multilabel", "imbalanced"},
    "bahasa":     {"en", "id"},
    "arsitektur": {"meanpool", "cnn", "lstm", "gru"},
    "embedding":  {"acak", "word2vec"},
}


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")


def validasi(cfg):
    cfg = dict(cfg)
    for k, sah in PILIHAN.items():
        if cfg.get(k) not in sah:
            raise ValueError(f'CFG["{k}"] = {cfg.get(k)!r} tidak sah. Pilihan: {sorted(sah)}')
    if cfg["embedding"] == "word2vec":
        try:
            import gensim  # noqa: F401
        except ImportError:
            print("  [config disesuaikan] gensim tidak ada -> embedding kembali ke 'acak'")
            cfg["embedding"] = "acak"
    if cfg["freeze_emb"] and cfg["embedding"] == "acak":
        print("  [config disesuaikan] membekukan embedding acak = tidak belajar -> freeze_emb=False")
        cfg["freeze_emb"] = False
    if cfg["case"] == "multilabel" and cfg["seimbangkan"]:
        print("  [config disesuaikan] bobot kelas belum didukung untuk multilabel -> diabaikan")
        cfg["seimbangkan"] = False
    return cfg

torch 2.6.0+cu124 | device: cuda | NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
def _baca_tabel(path, cfg):
    path = Path(path)
    sep = cfg["sep"] or ("\t" if path.suffix.lower() in {".tsv", ".tab"} else ",")
    for enc in ([cfg["encoding"]] if cfg["encoding"] else ["utf-8", "latin-1", "cp1252"]):
        try:
            return pd.read_csv(path, sep=sep, header=cfg["header"], encoding=enc,
                               engine="python", on_bad_lines="skip")
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise ValueError(f"gagal membaca {path}")


def _pilih_kolom(df, cfg):
    tc, lc = cfg["text_col"], cfg["label_col"]
    if tc is None:
        kand = [c for c in df.columns if df[c].map(lambda v: isinstance(v, str)).mean() > 0.5]
        tc = max(kand, key=lambda c: df[c].astype(str).str.len().mean())
    if lc is None:
        batas = 200 if cfg["case"] == "multilabel" else 20
        kand = [(df[c].nunique(dropna=True), c) for c in df.columns
                if c != tc and 2 <= df[c].nunique(dropna=True) <= batas]
        lc = min(kand)[1]
    return tc, lc


def _rapikan(df, cfg):
    df = df.copy()
    df["text"] = df["text"].astype(str).str.strip()
    if cfg["case"] != "multilabel":
        df["label"] = df["label"].apply(lambda v: v.strip().lower() if isinstance(v, str) else v)
        if cfg["label_map"]:
            df["label"] = df["label"].map(cfg["label_map"]).fillna(df["label"])
    df = df[df["text"].str.len() >= 3]
    return df.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"]).reset_index(drop=True)


def muat(cfg):
    if cfg["sumber"] == "dataframe":
        df = DF_MANUAL.copy()
        tc, lc = _pilih_kolom(df, cfg)
        return _rapikan(df.rename(columns={tc: "text", lc: "label"})[["text", "label"]], cfg), None
    if cfg["sumber"] == "folder_txt":
        from sklearn.datasets import load_files
        b = load_files(cfg["path"], encoding="utf-8", decode_error="replace")
        df = pd.DataFrame({"text": b.data, "label": [b.target_names[i] for i in b.target]})
        return _rapikan(df, cfg), None
    if cfg["sumber"] == "train_test":
        out = []
        for p in (cfg["path_train"], cfg["path_test"]):
            d = _baca_tabel(p, cfg)
            tc, lc = _pilih_kolom(d, cfg)
            out.append(_rapikan(d.rename(columns={tc: "text", lc: "label"})[["text", "label"]], cfg))
        tr, te = out
        bocor = set(tr["text"]) & set(te["text"])
        if bocor:
            print(f"  [peringatan] {len(bocor)} dokumen bocor train<->test, dibuang dari test")
            te = te[~te["text"].isin(bocor)].reset_index(drop=True)
        return tr, te
    d = _baca_tabel(cfg["path"], cfg)
    tc, lc = _pilih_kolom(d, cfg)
    d = _rapikan(d.rename(columns={tc: "text", lc: "label"})[["text", "label"]], cfg)
    if cfg["sample"] and cfg["sample"] < len(d):
        strat = d["label"] if cfg["case"] != "multilabel" else None
        d, _ = train_test_split(d, train_size=cfg["sample"], stratify=strat,
                                random_state=cfg["random_state"])
        d = d.reset_index(drop=True)
    return d, None

---
## §4 · Mesin: preprocessing → vocab → tensor

In [4]:
STOP_EN = {"i","me","my","we","our","you","your","he","she","it","they","them","this","that",
           "is","are","was","were","be","the","a","an","and","but","or","of","at","by","for",
           "with","to","from","in","on","so","than","too","very","just","now"}
STOP_ID = {"yang","dan","di","ke","dari","ini","itu","untuk","dengan","pada","adalah","ada",
           "saya","kamu","dia","kami","kita","mereka","akan","sudah","juga","atau","karena","saja"}
NEGASI_EN = {"no","not","never","nor","cannot"}
NEGASI_ID = {"tidak","bukan","tanpa","jangan","belum","kurang"}
SLANG_ID = {"gk":"tidak","ga":"tidak","gak":"tidak","yg":"yang","tdk":"tidak","bgt":"banget",
            "udh":"sudah","blm":"belum","krn":"karena","aja":"saja"}
PAD, UNK = 0, 1

try:
    from nltk.stem import PorterStemmer
    _stem_en = PorterStemmer()
except Exception:
    _stem_en = None
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    _stem_id = StemmerFactory().create_stemmer()
except Exception:
    _stem_id = None


def buat_tokenizer(cfg):
    stop = (STOP_EN if cfg["bahasa"] == "en" else STOP_ID)
    if cfg["jaga_negasi"]:
        stop = stop - (NEGASI_EN if cfg["bahasa"] == "en" else NEGASI_ID)

    def tok(teks):
        t = str(teks)
        if cfg["lowercase"]:
            t = t.lower()
        if cfg["mask_entity"]:
            t = re.sub(r"http\S+|www\.\S+", " urltoken ", t)
            t = re.sub(r"@\w+", " usertoken ", t)
            t = re.sub(r"\d+", " numtoken ", t)
        kata = re.findall(r"[a-zA-Z]+", t)
        if cfg["normalisasi_slang"]:
            kata = [SLANG_ID.get(w, w) for w in kata]
        if cfg["buang_stopword"]:
            kata = [w for w in kata if w not in stop]
        if cfg["stemming"]:
            if cfg["bahasa"] == "en" and _stem_en:
                kata = [_stem_en.stem(w) for w in kata]
            elif cfg["bahasa"] == "id" and _stem_id:
                kata = [_stem_id.stem(w) for w in kata]
        return kata or ["kosongtoken"]

    return tok


def bangun_vocab(daftar_teks, tok, cfg):
    c = Counter(w for t in daftar_teks for w in tok(t))
    kata = [w for w, n in c.most_common(cfg["max_vocab"]) if n >= cfg["min_freq"]]
    itos = ["<pad>", "<unk>"] + kata
    return {w: i for i, w in enumerate(itos)}, itos

In [5]:
class DatasetTeks(Dataset):
    def __init__(self, teks, y, tok, stoi, max_len):
        self.ids = [[stoi.get(w, UNK) for w in tok(t)][:max_len] or [UNK] for t in teks]
        self.y = y

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return torch.tensor(self.ids[i], dtype=torch.long), self.y[i]


def buat_collate(cfg, multilabel):
    min_len = max(cfg["kernel"]) if cfg["arsitektur"] == "cnn" else 1

    def collate(batch):
        urut, label = zip(*batch)
        panjang = torch.tensor([len(x) for x in urut], dtype=torch.long)
        padded = pad_sequence(urut, batch_first=True, padding_value=PAD)
        if padded.size(1) < min_len:
            padded = F.pad(padded, (0, min_len - padded.size(1)), value=PAD)
        y = (torch.tensor(np.array(label), dtype=torch.float) if multilabel
             else torch.tensor(label, dtype=torch.long))
        return padded, panjang, y

    return collate

---
## §5 · Mesin: arsitektur

In [6]:
class Meanpool(nn.Module):
    def __init__(self, n_vocab, cfg, n_out):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, cfg["dim"], padding_idx=PAD)
        self.drop = nn.Dropout(cfg["dropout"])
        self.fc = nn.Linear(cfg["dim"], n_out)

    def forward(self, x, panjang):
        e = self.emb(x)
        mask = (x != PAD).unsqueeze(-1).float()
        return self.fc(self.drop((e * mask).sum(1) / mask.sum(1).clamp(min=1)))


class CNNTeks(nn.Module):
    def __init__(self, n_vocab, cfg, n_out):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, cfg["dim"], padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Conv1d(cfg["dim"], cfg["n_filter"], k)
                                    for k in cfg["kernel"]])
        self.drop = nn.Dropout(cfg["dropout"])
        self.fc = nn.Linear(cfg["n_filter"] * len(cfg["kernel"]), n_out)

    def forward(self, x, panjang=None):
        e = self.emb(x).transpose(1, 2)
        f = [F.relu(c(e)).max(dim=2).values for c in self.convs]
        return self.fc(self.drop(torch.cat(f, dim=1)))


class RNNTeks(nn.Module):
    def __init__(self, n_vocab, cfg, n_out):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, cfg["dim"], padding_idx=PAD)
        kelas_rnn = nn.LSTM if cfg["arsitektur"] == "lstm" else nn.GRU
        self.rnn = kelas_rnn(cfg["dim"], cfg["hidden"], batch_first=True,
                             bidirectional=cfg["bidirectional"])
        self.bi = cfg["bidirectional"]
        self.drop = nn.Dropout(cfg["dropout"])
        self.fc = nn.Linear(cfg["hidden"] * (2 if self.bi else 1), n_out)

    def forward(self, x, panjang):
        packed = pack_padded_sequence(self.emb(x), panjang.cpu(), batch_first=True,
                                      enforce_sorted=False)
        keluar = self.rnn(packed)[1]
        h = keluar[0] if isinstance(keluar, tuple) else keluar     # LSTM -> (h, c); GRU -> h
        h = torch.cat([h[-2], h[-1]], dim=1) if self.bi else h[-1]
        return self.fc(self.drop(h))


def buat_model(cfg, n_vocab, n_out, matriks_emb=None):
    kelas = {"meanpool": Meanpool, "cnn": CNNTeks, "lstm": RNNTeks, "gru": RNNTeks}[cfg["arsitektur"]]
    m = kelas(n_vocab, cfg, n_out)
    if matriks_emb is not None:
        m.emb = nn.Embedding.from_pretrained(torch.tensor(matriks_emb),
                                             freeze=cfg["freeze_emb"], padding_idx=PAD)
    return m


def latih_word2vec(teks, tok, itos, cfg):
    from gensim.models import Word2Vec
    w2v = Word2Vec(sentences=[tok(t) for t in teks], vector_size=cfg["dim"], window=5,
                   min_count=1, sg=1, epochs=10, workers=1, seed=cfg["random_state"])
    M = np.random.normal(0, 0.1, (len(itos), cfg["dim"])).astype("float32")
    M[PAD] = 0
    ketemu = 0
    for i, w in enumerate(itos):
        if w in w2v.wv:
            M[i] = w2v.wv[w]; ketemu += 1
    print(f"  word2vec: {ketemu}/{len(itos)} kata dapat vektor")
    return M

---
## §6 · Mesin: training loop + evaluasi

In [7]:
@torch.no_grad()
def prediksi_loader(model, loader, multilabel):
    model.eval()
    P, Y = [], []
    for x, panjang, y in loader:
        logits = model(x.to(device), panjang.to(device))
        P.append((torch.sigmoid(logits) > 0.5).int().cpu() if multilabel
                 else logits.argmax(1).cpu())
        Y.append(y.cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy()


def jalankan(cfg, verbose=True):
    cfg = validasi(cfg)
    p = print if verbose else (lambda *a, **k: None)
    set_seed(cfg["random_state"])
    multilabel = cfg["case"] == "multilabel"

    # --- data
    df_tr, df_te = muat(cfg)
    if df_te is None:
        strat = df_tr["label"] if not multilabel else None
        df_tr, df_te = train_test_split(df_tr, test_size=cfg["test_size"], stratify=strat,
                                        random_state=cfg["random_state"])
    strat = df_tr["label"] if not multilabel else None
    df_tr, df_val = train_test_split(df_tr, test_size=cfg["val_size"], stratify=strat,
                                     random_state=cfg["random_state"])
    df_tr, df_val, df_te = [d.reset_index(drop=True) for d in (df_tr, df_val, df_te)]
    p(f"data      : train {len(df_tr)} | val {len(df_val)} | test {len(df_te)}")

    # --- label
    if multilabel:
        from sklearn.preprocessing import MultiLabelBinarizer
        pisah = lambda s: [x.strip() for x in str(s).split(cfg["pemisah_label"]) if x.strip()]
        mlb = MultiLabelBinarizer()
        ytr = mlb.fit_transform(df_tr["label"].map(pisah)).astype("float32")
        yval = mlb.transform(df_val["label"].map(pisah)).astype("float32")
        yte = mlb.transform(df_te["label"].map(pisah)).astype("float32")
        kelas, n_out = list(mlb.classes_), len(mlb.classes_)
    else:
        mlb = None
        kelas = sorted(set(map(str, df_tr["label"])))
        l2i = {c: i for i, c in enumerate(kelas)}
        ytr  = np.array([l2i[str(v)] for v in df_tr["label"]])
        yval = np.array([l2i[str(v)] for v in df_val["label"]])
        yte  = np.array([l2i[str(v)] for v in df_te["label"]])
        n_out = len(kelas)
        p(f"kelas     : {kelas}")

    # --- teks -> tensor
    tok = buat_tokenizer(cfg)
    stoi, itos = bangun_vocab(df_tr["text"], tok, cfg)          # vocab dari TRAIN saja
    p(f"vocab     : {len(itos)} kata")
    collate = buat_collate(cfg, multilabel)
    mk = lambda d, y, sh: DataLoader(DatasetTeks(d["text"], y, tok, stoi, cfg["max_len"]),
                                     batch_size=cfg["batch"], shuffle=sh, collate_fn=collate)
    dl_tr, dl_val, dl_te = mk(df_tr, ytr, True), mk(df_val, yval, False), mk(df_te, yte, False)

    # --- model
    M = latih_word2vec(df_tr["text"], tok, itos, cfg) if cfg["embedding"] == "word2vec" else None
    model = buat_model(cfg, len(itos), n_out, M).to(device)
    p(f"model     : {cfg['arsitektur']} | emb={cfg['embedding']}"
      f"{' (beku)' if cfg['freeze_emb'] else ''} | "
      f"{sum(q.numel() for q in model.parameters() if q.requires_grad):,} parameter")

    # --- loss
    if multilabel:
        criterion = nn.BCEWithLogitsLoss()
    elif cfg["seimbangkan"]:
        n = np.bincount(ytr, minlength=n_out)
        bobot = torch.tensor(len(ytr) / (n_out * np.maximum(n, 1)), dtype=torch.float, device=device)
        p(f"bobot kls : {bobot.cpu().numpy().round(2)}")
        criterion = nn.CrossEntropyLoss(weight=bobot)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"],
                                 weight_decay=cfg["weight_decay"])
    rata = "micro" if multilabel else "macro"

    # --- training loop
    terbaik, state = -1, None
    for ep in range(1, cfg["epochs"] + 1):
        model.train()
        total, t0 = 0.0, time.time()
        for x, panjang, y in dl_tr:
            x, panjang, y = x.to(device), panjang.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x, panjang), y)
            loss.backward()
            optimizer.step()
            total += loss.item() * y.size(0)
        pv, yv = prediksi_loader(model, dl_val, multilabel)
        f1v = f1_score(yv, pv, average=rata, zero_division=0)
        if f1v > terbaik:
            terbaik = f1v
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        p(f"  epoch {ep:2d}  loss {total/len(dl_tr.dataset):.4f}  val_f1 {f1v:.4f}"
          f"  ({time.time()-t0:.1f}s)")
    model.load_state_dict(state)

    # --- evaluasi
    pred, ytrue = prediksi_loader(model, dl_te, multilabel)
    f1 = f1_score(ytrue, pred, average=rata, zero_division=0)
    akurasi = accuracy_score(ytrue, pred) if not multilabel else np.nan
    p(f"\nTEST: f1_{rata} {f1:.3f}" + (f" | akurasi {akurasi:.3f}" if not multilabel else ""))
    p(classification_report(ytrue, pred, target_names=[str(k) for k in kelas], zero_division=0))
    if not multilabel:
        p(pd.DataFrame(confusion_matrix(ytrue, pred),
                       index=["true_" + str(k) for k in kelas],
                       columns=["pred_" + str(k) for k in kelas]).to_string())

    return {"model": model, "cfg": cfg, "f1": f1, "akurasi": akurasi, "kelas": kelas,
            "stoi": stoi, "itos": itos, "tok": tok, "pred": pred, "y_test": ytrue,
            "df_train": df_tr, "df_test": df_te, "val_f1": terbaik, "mlb": mlb}

---
## §7 · JALANKAN

In [8]:
hasil = jalankan(CFG)

data      : train 2040 | val 360 | test 800
kelas     : ['negatif', 'positif']
vocab     : 1773 kata


model     : meanpool | emb=acak | 177,502 parameter


  epoch  1  loss 0.7063  val_f1 0.5300  (0.3s)
  epoch  2  loss 0.6747  val_f1 0.5833  (0.1s)
  epoch  3  loss 0.6659  val_f1 0.5994  (0.1s)


  epoch  4  loss 0.6517  val_f1 0.6075  (0.1s)
  epoch  5  loss 0.6398  val_f1 0.6288  (0.1s)
  epoch  6  loss 0.6320  val_f1 0.6328  (0.1s)

TEST: f1_macro 0.635 | akurasi 0.635
              precision    recall  f1-score   support

     negatif       0.64      0.63      0.63       400
     positif       0.63      0.64      0.64       400

    accuracy                           0.64       800
   macro avg       0.64      0.64      0.63       800
weighted avg       0.64      0.64      0.63       800

              pred_negatif  pred_positif
true_negatif           252           148
true_positif           144           256


---
## §8 · Bandingkan beberapa config

In [9]:
VARIAN = [
    ("meanpool + emb acak",        {}),
    ("cnn (Kim 2014)",             {"arsitektur": "cnn"}),
    ("lstm bidirectional",         {"arsitektur": "lstm"}),
    ("gru",                        {"arsitektur": "gru"}),
    ("meanpool + word2vec (beku)", {"embedding": "word2vec", "freeze_emb": True}),
    ("meanpool + word2vec (setel)",{"embedding": "word2vec", "freeze_emb": False}),
    ("cnn + word2vec (setel)",     {"arsitektur": "cnn", "embedding": "word2vec"}),
]

baris = []
for nama, ubah in VARIAN:
    try:
        r = jalankan({**CFG, **ubah}, verbose=False)
        baris.append((nama, round(r["val_f1"], 3), round(r["f1"], 3)))
    except Exception as e:
        baris.append((nama, "gagal", f"{type(e).__name__}: {str(e)[:40]}"))

print(pd.DataFrame(baris, columns=["konfigurasi", "f1 val", "f1 test"]).to_string(index=False))

  word2vec: 1771/1773 kata dapat vektor


  word2vec: 1771/1773 kata dapat vektor


  word2vec: 1771/1773 kata dapat vektor


                konfigurasi  f1 val  f1 test
        meanpool + emb acak   0.633    0.635
             cnn (Kim 2014)   0.694    0.658
         lstm bidirectional   0.691    0.651
                        gru   0.711    0.645
 meanpool + word2vec (beku)   0.596    0.590
meanpool + word2vec (setel)   0.722    0.714
     cnn + word2vec (setel)   0.733    0.710


---
## §9 · Pembanding TF-IDF  ·  jangan dilewati

Angka neural tidak berarti apa-apa tanpa baseline klasik di sebelahnya.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

if hasil["cfg"]["case"] != "multilabel":
    tr, te = hasil["df_train"], hasil["df_test"]
    klasik = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                       ("clf", LogisticRegression(max_iter=1000))])
    klasik.fit(tr["text"], tr["label"].astype(str))
    pk = klasik.predict(te["text"])
    kelas = hasil["kelas"]
    l2i = {c: i for i, c in enumerate(kelas)}
    f1k = f1_score(hasil["y_test"], [l2i[str(v)] for v in pk], average="macro", zero_division=0)
    print(f"PyTorch ({hasil['cfg']['arsitektur']}, emb={hasil['cfg']['embedding']}):"
          f" f1_macro = {hasil['f1']:.3f}")
    print(f"TF-IDF + LogReg (sklearn)              : f1_macro = {f1k:.3f}")
    print("\n->", "neural menang" if hasil["f1"] > f1k else
          "model klasik menang -- wajar untuk data sebesar ini")
else:
    print("(pembanding TF-IDF dilewati untuk case multilabel)")

PyTorch (meanpool, emb=acak): f1_macro = 0.635
TF-IDF + LogReg (sklearn)              : f1_macro = 0.677

-> model klasik menang -- wajar untuk data sebesar ini


---
## §10 · Simpan model & prediksi teks baru

In [11]:
torch.save({"state_dict": hasil["model"].state_dict(), "itos": hasil["itos"],
            "kelas": hasil["kelas"], "cfg": hasil["cfg"]}, "model_latihan.pt")

ckpt = torch.load("model_latihan.pt", map_location=device, weights_only=False)
cfg2, itos2, kelas2 = ckpt["cfg"], ckpt["itos"], ckpt["kelas"]
stoi2 = {w: i for i, w in enumerate(itos2)}
model2 = buat_model(cfg2, len(itos2), len(kelas2)).to(device)
model2.load_state_dict(ckpt["state_dict"])
model2.eval()
tok2 = buat_tokenizer(cfg2)


@torch.no_grad()
def prediksi_teks(daftar):
    ids = [torch.tensor([stoi2.get(w, UNK) for w in tok2(t)][:cfg2["max_len"]] or [UNK])
           for t in daftar]
    panjang = torch.tensor([len(i) for i in ids])
    x = pad_sequence(ids, batch_first=True, padding_value=PAD)
    minimal = max(cfg2["kernel"]) if cfg2["arsitektur"] == "cnn" else 1
    if x.size(1) < minimal:
        x = F.pad(x, (0, minimal - x.size(1)), value=PAD)
    logits = model2(x.to(device), panjang.to(device))
    if cfg2["case"] == "multilabel":
        ada = (torch.sigmoid(logits) > 0.5).cpu().numpy()
        return [(t, [kelas2[j] for j in np.where(r)[0]]) for t, r in zip(daftar, ada)]
    prob = F.softmax(logits, dim=1).cpu().numpy()
    return [(t, kelas2[int(r.argmax())], float(r.max())) for t, r in zip(daftar, prob)]


for baris in prediksi_teks(["absolutely love this, best day ever thanks",
                            "worst service ever, i hate it, so disappointed"]):
    print(" ", baris)

  ('absolutely love this, best day ever thanks', 'positif', 0.6460091471672058)
  ('worst service ever, i hate it, so disappointed', 'negatif', 0.8221860527992249)


---
## Resep cepat per skenario

| Kalau dosen bilang… | Ubah di CFG |
|---|---|
| "implementasikan CNN untuk klasifikasi teks" | `"arsitektur": "cnn"` |
| "pakai LSTM/RNN" | `"arsitektur": "lstm"` (atau `"gru"`) |
| "pakai word embedding pretrained" | `"embedding": "word2vec"`, `"freeze_emb": False` |
| "bandingkan dengan dan tanpa pretrained" | jalankan §8 |
| "datanya bahasa Indonesia" | `"bahasa": "id"`, `"normalisasi_slang": True` |
| "kelasnya ada 4" | `"case": "multiclass"` + ganti `path` |
| "satu dokumen bisa banyak label" | `"case": "multilabel"`, `"label_col": "labels"` |
| "datanya timpang" | `"case": "imbalanced"`, `"seimbangkan": True` |
| "modelnya overfit" | naikkan `dropout`, kurangi `epochs`, isi `weight_decay` (mis. 1e-4) |
| "terlalu lambat" | turunkan `max_len`, `epochs`, atau pakai `"arsitektur": "meanpool"` |
| "harus ada baseline" | §9 sudah menghitungnya otomatis |